In [1]:
import numpy as np
import random

In [2]:
# ==========================================
# 1. THE GRID ENVIRONMENT
# ==========================================
class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.goal = size - 1
        self.state = 0

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):
        # Action 0: Left, Action 1: Right
        if action == 1:
            self.state = min(self.size - 1, self.state + 1)
        elif action == 0:
            self.state = max(0, self.state - 1)
        
        reward = 10 if self.state == self.goal else -1
        done = self.state == self.goal
        return self.state, reward, done

In [3]:
# ==========================================
# 2. THE FEDERATED AGENT (Client)
# ==========================================
class RLAgent:
    def __init__(self, agent_id, state_size, action_size):
        self.agent_id = agent_id
        self.q_table = np.zeros((state_size, action_size))
        self.env = GridWorld(size=state_size)

    def local_train(self, global_q_table, episodes=10, alpha=0.1, gamma=0.9, epsilon=0.1):
        # 1. Receive Global Knowledge
        if global_q_table is not None:
            self.q_table = np.copy(global_q_table)
        
        # 2. Local Q-Learning
        for _ in range(episodes):
            state = self.env.reset()
            done = False
            while not done:
                # Epsilon-greedy action selection
                if random.uniform(0, 1) < epsilon:
                    action = random.choice([0, 1])
                else:
                    action = np.argmax(self.q_table[state])
                
                next_state, reward, done = self.env.step(action)
                
                # Q-Value Update: Q(s,a) = Q(s,a) + alpha * [R + gamma * maxQ(s',a') - Q(s,a)]
                best_next_action = np.argmax(self.q_table[next_state])
                td_target = reward + gamma * self.q_table[next_state][best_next_action]
                self.q_table[state][action] += alpha * (td_target - self.q_table[state][action])
                
                state = next_state
                
        return self.q_table

In [4]:
# ==========================================
# 3. THE FEDERATED SERVER (Aggregation)
# ==========================================
class RLServer:
    def __init__(self, state_size, action_size):
        self.global_q_table = np.zeros((state_size, action_size))

    def aggregate(self, agent_q_tables):
        """Average the Q-values from all agents."""
        self.global_q_table = np.mean(agent_q_tables, axis=0)

In [5]:
# ==========================================
# 4. SIMULATION EXECUTION
# ==========================================
def run_federated_rl(num_agents=3, rounds=50):
    state_size = 6
    action_size = 2 # Left, Right
    
    server = RLServer(state_size, action_size)
    agents = [RLAgent(i, state_size, action_size) for i in range(num_agents)]
    
    print(f"Starting Federated RL: {num_agents} agents learning to reach state {state_size-1}...")
    
    for r in range(rounds):
        local_tables = []
        for agent in agents:
            # Each agent trains for 5 episodes locally
            q_table = agent.local_train(server.global_q_table, episodes=5)
            local_tables.append(q_table)
        
        # Server averages local Q-tables
        server.aggregate(local_tables)
        
        if (r + 1) % 10 == 0:
            # Display the 'policy' learned by the global model
            policy = [np.argmax(server.global_q_table[s]) for s in range(state_size - 1)]
            # Convert 0/1 to arrows for readability
            arrows = ['→' if p == 1 else '←' for p in policy]
            print(f"Round {r+1:02d} | Global Policy: {' '.join(arrows)} | Goal Reached!")

In [6]:
if __name__ == "__main__":
    run_federated_rl()

Starting Federated RL: 3 agents learning to reach state 5...
Round 10 | Global Policy: → → → → → | Goal Reached!
Round 20 | Global Policy: → → → → → | Goal Reached!
Round 30 | Global Policy: → → → → → | Goal Reached!
Round 40 | Global Policy: → → → → → | Goal Reached!
Round 50 | Global Policy: → → → → → | Goal Reached!
